In [0]:
from pyspark.sql.functions import current_timestamp

def ingest_to_bronze(file_name, table_name):
    path = f"/Volumes/visagio_rocket_lab/bronze/olist_raw/{file_name}"
    
    #Le o csv
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(path)
    
    #Adicionando o timestamp_ingestion 
    df_with_timestamp = df.withColumn("timestamp_ingestion", current_timestamp())
    
    #Salvando na bronze 
    df_with_timestamp.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"visagio_rocket_lab.bronze.{table_name}")
    
    print(f"Sucesso: {table_name}")

ingest_to_bronze("olist_customers_dataset.csv", "tb_customers")
ingest_to_bronze("olist_orders_dataset.csv", "tb_orders")
ingest_to_bronze("olist_order_items_dataset.csv", "tb_order_items")
ingest_to_bronze("olist_products_dataset.csv", "tb_products")
ingest_to_bronze("olist_sellers_dataset.csv", "tb_sellers")
ingest_to_bronze("olist_order_payments_dataset.csv", "tb_order_payments")
ingest_to_bronze("olist_order_reviews_dataset.csv", "tb_order_reviews")
ingest_to_bronze("olist_geolocation_dataset.csv", "tb_geolocalizacao")
ingest_to_bronze("product_category_name_translation.csv", "tb_product_category_name_translation")

Sucesso: tb_customers
Sucesso: tb_orders
Sucesso: tb_order_items
Sucesso: tb_products
Sucesso: tb_sellers
Sucesso: tb_order_payments
Sucesso: tb_order_reviews
Sucesso: tb_geolocalizacao
Sucesso: tb_product_category_name_translation


In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp

#Criando os parâmetros do notebook 
dbutils.widgets.text("data_inicio_formatada", "01-01-2016")
dbutils.widgets.text("data_fim_formatada", "12-31-2018")

# Capturando os valores dos widgets
data_ini = dbutils.widgets.get("data_inicio_formatada")
data_fim = dbutils.widgets.get("data_fim_formatada")

#puxando api
url = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_ini}'&@dataFinalCotacao='{data_fim}'&"
    f"$select=dataHoraCotacao,cotacaoCompra&$format=json"
)


try:
    response = requests.get(url, timeout=15)
    response.raise_for_status() #Verifica se a conexão deu certo
    
    
    dados_json = response.json()['value']
    
    #Transformando em DataFrame do Spark via Pandas 
    df_dolar = spark.createDataFrame(pd.DataFrame(dados_json))
    
    #Adicionando o timestamp_ingestion
    df_dolar_final = df_dolar.withColumn("timestamp_ingestion", current_timestamp())
    
  
    df_dolar_final.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("visagio_rocket_lab.bronze.tb_cotacao_dolar")
    
    print("Sucesso: bronze.tb_cotacao_dolar criada!")

except Exception as e:
    print(f"Erro ao extrair dados da API: {e}")

Sucesso: bronze.tb_cotacao_dolar criada!
